# Omni multi-vector retrieval trên YouCook2 (UEMR P3 + P4, chưa có Athena)

Encode ~500 video YouCook2 bằng OmniRetriever-7B (adapter fine-tune), truy vấn tương tác, chấm nhanh R@1/5/10, MedR text→video cho `global`, `gt`, `uni_M_gt` (+ `event_single`).

* **Encode bằng đúng pipeline của `eval_youcookii.py`** (`LazySupervisedDataset`: decord, 8 frame, 50176 px giữ tỉ lệ, audio cắt từ mp4 rồi center-crop 8 s, `<|AUDIO|>` nhân đôi). Không dùng `omniretriever.cli extract` vì CLI tiền xử lý khác (PyAV, center-crop vuông, 336 px). Text = `caption<|im_end|>` → hidden state lớp cuối ở token cuối (nhánh label của eval), **không** phải `OmniRetriever.encode_text`.
* Cache khử trùng lặp theo `(video, ts, te)`; thêm partition mới (ví dụ proposals của Athena) chỉ encode đoạn chưa có. Mọi cell chạy lại được an toàn.

**Thứ tự chạy**
1. Cell 1–4 (setup).
2. Cell 5–7: unit test, dry-run, test collect/evaluate bằng npz giả. Không cần GPU, không nạp model.
3. Cell 8–15 với `SMOKE = True` (5 video, encode thật).
4. Smoke ổn → đặt `SMOKE = False` ở cell 4, chạy lại cell 4 rồi 8–15 (embedding của smoke được dùng lại).
5. Cell 16–17: giải phóng bộ nhớ, query tương tác.

**Runtime:** A100.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Lấy code và cài thư viện

`CODE_SOURCE = 'git'` (đã push Fushion lên GitHub, sửa `FUSHION_GIT`) hoặc `'drive'` (đã upload cả thư mục `Code/Fushion` lên `FUSHION_DRIVE`). Omni-fix nhánh `chien`, không sửa gì trong đó. `av`, `decord`, `pytest` là bắt buộc.

In [ ]:
import os, sys, shutil, json, subprocess
CODE_SOURCE   = 'drive'   # 'git' | 'drive'
FUSHION_GIT   = 'https://github.com/<user>/Fushion.git'          # <-- sửa nếu dùng git
FUSHION_DRIVE = '/content/drive/MyDrive/uemr/code/Fushion'        # <-- thư mục Fushion đã upload
FUSHION   = '/content/code/Fushion'
OMNI_REPO = '/content/code/Omni-fix'

os.makedirs('/content/code', exist_ok=True)
if CODE_SOURCE == 'git':
    if os.path.isdir(f'{FUSHION}/.git'):
        !git -C {FUSHION} pull -q
    else:
        shutil.rmtree(FUSHION, ignore_errors=True)
        !git clone -q {FUSHION_GIT} {FUSHION}
else:
    # copy lại mỗi lần để lấy bản mới nhất; bỏ cache/tmp
    shutil.rmtree(FUSHION, ignore_errors=True)
    shutil.copytree(FUSHION_DRIVE, FUSHION,
                    ignore=shutil.ignore_patterns('__pycache__', '.pytest_cache', 'tmp', '*.npz'))
assert os.path.isfile(f'{FUSHION}/omni_retrieval/search.py'), 'không thấy omni_retrieval trong ' + FUSHION

if not os.path.isdir(OMNI_REPO):
    !git clone -q -b chien --single-branch https://github.com/im-xiaoming/Omni-fix.git {OMNI_REPO}
!git -C {OMNI_REPO} log --oneline -1

if FUSHION not in sys.path:
    sys.path.insert(0, FUSHION)
os.environ['PYTHONPATH'] = FUSHION + ':' + os.environ.get('PYTHONPATH', '')

if not os.path.exists('/content/.omni_deps_ok'):
    %cd {OMNI_REPO}
    !pip install -q -r requirements.txt
    !pip install -q -r training/requirements.txt
    !pip uninstall -y -q torchao
    !pip install -q -U torchao av decord pytest
    open('/content/.omni_deps_ok', 'w').close()
%cd {FUSHION}
import av, decord  # noqa: F401  (báo lỗi ở đây, không phải sau khi nạp model)
print('OK')

## 3. Tải WAVE-7B, BEATs và adapter

Adapter fine-tune = thư mục `best` mà `Omni_training_uemr.ipynb` chọn trên dev (copy 2 file về `/content` để đọc nhanh). Adapter gốc chỉ tải khi `TAG = 'pt'` (cell 4). Không cần vá M-RoPE: pipeline eval dùng model vendored trong `training/qwenvl`.

In [ ]:
FT_DRIVE = '/content/drive/MyDrive/omniretriever/youcookii_ft_uemr/best'
FT_LOCAL = '/content/adapters/ft'
PT_LOCAL = '/content/adapters/omniretriever-7b'
WAVE     = '/content/WAVE_HOME/WAVE-7B/'
BEATS    = '/content/WAVE_HOME/BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt'

if not os.path.exists(WAVE + 'config.json'):
    !hf download nguyenminh04/omni-model --local-dir /content/WAVE_HOME --quiet
os.makedirs(FT_LOCAL, exist_ok=True)
for f in ('adapter_config.json', 'adapter_model.safetensors', 'SOURCE.json'):
    if os.path.exists(f'{FT_DRIVE}/{f}') and not os.path.exists(f'{FT_LOCAL}/{f}'):
        shutil.copy2(f'{FT_DRIVE}/{f}', f'{FT_LOCAL}/{f}')
src = f'{FT_LOCAL}/SOURCE.json'
print('adapter fine-tune lấy từ:', json.load(open(src)) if os.path.exists(src) else '(không có SOURCE.json)')
for p in (WAVE + 'config.json', BEATS, f'{FT_LOCAL}/adapter_model.safetensors'):
    print('%-6s %s' % ('OK' if os.path.exists(p) else 'THIẾU', p))

## 4. Cấu hình

`TAG = 'ft'` trước; chạy lại với `TAG = 'pt'` cho cột (a) (cùng manifest, khác thư mục cache). `EXTRA_PARTITIONS` thêm partition từ JSON `video_id → [[ts, te], ...]`, ví dụ `{'uniav_pred': '/content/drive/MyDrive/uemr/partitions/uniav_pred.json'}`.

Sanity check ở cell 15 so với `best_val.json` (`ft`) hoặc `zeroshot_val.json` (`pt`) do notebook fine-tune ghi ra.

In [ ]:
from omni_retrieval.config import Config

TAG      = 'ft'        # 'ft' | 'pt'
SMOKE    = True        # True: 5 video encode thật; False: chạy đầy đủ
N_VIDEOS = 500         # tối đa 394 val + 104 dev = 498 (không dùng video train)
SEED     = 0
EXTRA_PARTITIONS = {}  # {'uniav_pred': '/content/drive/.../uniav_pred.json'}
RUN = '/content/drive/MyDrive/omniretriever/youcookii_ft_uemr'

def make_cfg(smoke):
    return Config.colab(tag=TAG, smoke=smoke, n_videos=N_VIDEOS, seed=SEED,
                        extra_partitions=EXTRA_PARTITIONS, omni_repo=OMNI_REPO,
                        wave_path=WAVE, beats_path=BEATS, adapter_ft=FT_LOCAL, adapter_pt=PT_LOCAL,
                        ref_eval_json=f"{RUN}/eval/{'best_val' if TAG == 'ft' else 'zeroshot_val'}.json")

cfg = make_cfg(SMOKE)
if TAG == 'pt' and not os.path.exists(f'{PT_LOCAL}/adapter_model.safetensors'):
    !hf download YunzeLiu/OmniRetriever-7B --local-dir {PT_LOCAL} --quiet
ADAPTER = cfg.adapter
print('SMOKE    :', SMOKE)
print('adapter  :', ADAPTER)
print('cache    :', cfg.tag_dir, '| output:', cfg.out_dir)
print('reference:', cfg.ref_eval_json, '(có)' if os.path.exists(cfg.ref_eval_json) else '(KHÔNG có)')

## 5. Unit test

Vector giả: MaxSim, timestamp `argmax`, gom theo video, metric trên ví dụ tính tay, collect/evaluate trên store giả. Không cần GPU. Phải pass hết.

In [ ]:
!cd {FUSHION} && python -m pytest omni_retrieval/tests -q -p no:cacheprovider

## 6. Dry-run (cấu hình đầy đủ, không ghi gì, không nạp model)

`subset` → `partitions` → `manifest` trên metadata trên Drive; duration đọc từ header mp4 trên Drive (8 luồng, chỉ header). In số video, số đoạn mỗi partition, số clip sau khi khử trùng lặp và 3 dòng manifest mẫu.

In [ ]:
from omni_retrieval.manifest import prepare
dry = prepare(make_cfg(smoke=False), write=False)
print('3 dòng segs_todo mẫu:')
for r in dry['segs_todo'][:3]:
    print(' ', json.dumps(r, ensure_ascii=False))
print('2 dòng caps_todo mẫu:')
for r in dry['caps_todo'][:2]:
    print(' ', json.dumps(r, ensure_ascii=False))

## 7. Test collect/evaluate với npz giả

Sinh `fake_av.npz` / `fake_text.npz` đúng format key `<id>__av` / `<id>__text` như CLI ghi ra, cho đúng bảng đoạn và caption của dry-run; vector text = vector clip GT của chính nó + nhiễu. Rồi chạy `collect` + `evaluate`: text→clip phải ≈ 100 % và `gt` phải thắng `global`.

In [ ]:
from omni_retrieval.fake import run_fake_pipeline
_ = run_fake_pipeline(dry, '/content/fake_test')

## 8. Chọn subset

Theo `SMOKE` ở cell 4. Query = caption val; gallery = mọi video val + video dev làm distractor. Lưu `common/subset_videos[_smoke].json`; chạy lại không probe lại.

In [ ]:
from omni_retrieval.subset import build_subset
subset = build_subset(cfg)
n_val = sum(r['is_query_source'] for r in subset)
print(f'SMOKE={SMOKE}: {len(subset)} video = {n_val} val (nguồn query) + {len(subset) - n_val} dev (distractor) -> {cfg.subset_path}')

## 9. Copy video về `/content/videos`

Đọc thẳng từ Drive rất chậm. `shutil.copy2` từng file, tối đa 8 luồng, không đọc nội dung vào RAM; bỏ qua file đã có đủ dung lượng.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
os.makedirs(cfg.video_root, exist_ok=True)

def copy_one(vid):
    src, dst = f'{cfg.video_src}/{vid}.mp4', f'{cfg.video_root}/{vid}.mp4'
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        return 0
    shutil.copy2(src, dst + '.part')
    os.replace(dst + '.part', dst)
    return 1

with ThreadPoolExecutor(8) as pool:
    copied = sum(pool.map(copy_one, [r['video_id'] for r in subset]))
print(f'đã copy {copied}, có sẵn {len(subset) - copied}')
!df -h /content | tail -1

## 10. Sinh manifest

Bảng đoạn của mọi partition → khử trùng lặp theo `(video, ts, te)` làm tròn 0.01 s → `segs_todo` chỉ gồm đoạn **chưa có** trong cache, `caps_todo` chỉ gồm caption chưa encode.

In [ ]:
prepared = prepare(cfg)
print(cfg.segs_todo)
print(cfg.caps_todo)

## 11. Ước lượng thời gian

Encode 20 clip (giữ lại trong cache, không phí), đo giây/clip, nhân với số clip còn lại.

In [ ]:
ENC = (f'cd {FUSHION} && python -m omni_retrieval.encode --omni-repo {OMNI_REPO} '
       f'--base-model {WAVE} --beats-path {BEATS} --adapter {ADAPTER} --video-root {cfg.video_root}')
timing = cfg.tag_dir / 'timing_20.json'
n_left = len(prepared['segs_todo'])
if n_left >= 20 and not timing.exists():
    !{ENC} --modality av --manifest {cfg.segs_todo} --store {cfg.av_store} --limit 20 --timing-out {timing}
if timing.exists():
    t = json.load(open(timing))
    spc = t['sec_per_clip']
    print(f"giây/clip (median): {spc:.2f} | nạp model: {t['model_load_sec']:.0f} s | lỗi: {t['failed']}")
    print(f'còn {n_left} clip -> khoảng {n_left * spc / 3600:.2f} giờ GPU (+ nạp model)')

## 12. Encode `av`

Batch 1 như eval; 4 worker decode song song với GPU. Ghi thẳng lên Drive theo chunk 200 clip. Phiên Colab ngắt: chạy lại cell 1–4, 10 rồi cell này, chỉ encode phần còn thiếu. Nếu worker bị kill vì RAM, thêm `--num-workers 0`.

In [ ]:
!{ENC} --modality av --manifest {cfg.segs_todo} --store {cfg.av_store} --num-workers 4
failed = cfg.av_store / 'failed.jsonl'
if failed.exists():
    print('CÓ CLIP LỖI, xem', failed)
    print(open(failed).read()[:2000])

## 13. Encode `text`

Caption val của subset, cùng nhánh text của eval (batch theo độ dài token, không padding).

In [ ]:
!{ENC} --modality text --manifest {cfg.caps_todo} --store {cfg.text_store} --batch-size 32

## 14. Gom cache

`segments.npz`, `text.npz`, `meta.json` trong `omni_cache/<tag>/` (smoke: `<tag>/smoke/`). Đọc store từng key, ghi fp16. Báo lỗi nếu thiếu embedding.

In [ ]:
from omni_retrieval.collect import collect
prepared = prepare(cfg, verbose=False)   # làm mới bảng sau khi encode
meta = collect(cfg, prepared)

## 15. Đánh giá

**Sanity check bắt buộc:** text→clip trên đoạn `gt` (gallery giống eval) phải khớp t2m R@1 của eval adapter; lệch > 1 điểm thì in cảnh báo lớn và dừng (smoke: chỉ in, không so được vì gallery nhỏ). Sau đó bảng text→video theo partition, lưu `results_<tag>.csv`.

**Smoke xong và ổn → sửa `SMOKE = False` ở cell 4, chạy lại cell 4 rồi 8–15.**

In [ ]:
from IPython.display import Markdown, display
from omni_retrieval.evaluate import evaluate, markdown, sanity_check, save_csv

sanity = sanity_check(cfg.out_dir, cfg.ref_eval_json, tol=1.0, strict=not SMOKE)
parts = ['global', 'event_single', 'uni_M_gt', 'gt'] + list(EXTRA_PARTITIONS)
rows = evaluate(cfg.out_dir, parts, agg='max')
display(Markdown(markdown(rows)))
save_csv(rows, cfg.out_dir / f'results_{TAG}.csv')
json.dump({'sanity': sanity, 'rows': rows}, open(cfg.out_dir / f'results_{TAG}.json', 'w'), indent=1, default=float)
print('->', cfg.out_dir / f'results_{TAG}.csv')

## 16. Giải phóng bộ nhớ trước khi nạp model

Encode chạy trong tiến trình con nên kernel chưa giữ model; cell này xoá các bảng lớn còn lại và dọn cache CUDA. Model đã nạp ở cell 17 thì **giữ lại** (không nạp lần hai), trừ khi `UNLOAD_MODEL = True`.

In [ ]:
import gc, torch
UNLOAD_MODEL = False
for name in ('dry', 'prepared', 'rows', 'meta', 'sanity', 'index', 'text'):
    globals().pop(name, None)
if UNLOAD_MODEL and 'MODEL' in globals():
    del MODEL, PROC
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'GPU đang dùng: {torch.cuda.memory_allocated() / 2**30:.1f} GiB')
print('model trong kernel:', 'có (sẽ dùng lại)' if 'MODEL' in globals() else 'chưa')

## 17. Query tương tác

* `LOAD_MODEL = False`: không cần GPU, lấy caption val thứ `CAPTION_INDEX` trong `text.npz` làm query.
* `LOAD_MODEL = True`: nạp model **một lần** (cùng adapter với cache, cùng nhánh text của eval) rồi gõ query; Enter trống để thoát. Lần nạp đầu tự kiểm tra: encode lại một caption trong cache, cosine phải ≈ 1.

Kết quả: `video_id`, điểm, `[t_s, t_e]` của đoạn khớp nhất, caption GT gần đoạn đó nhất, và clip cắt bằng ffmpeg.

In [ ]:
import numpy as np, pandas as pd
from IPython.display import Video, display
from omni_retrieval.search import SegmentIndex, tiou
from omni_retrieval.evaluate import load_text

PARTITION     = 'gt'   # 'global' | 'gt' | 'uni_M_gt' | 'event_single' | partition thêm
TOP_K         = 5
SHOW_CLIPS    = 2      # số clip đầu bảng nhúng vào để xem
LOAD_MODEL    = False
CAPTION_INDEX = 0

index = SegmentIndex.load(cfg.out_dir, PARTITION)
text  = load_text(cfg.out_dir)

def nearest_gt_caption(vid, seg):
    sel = np.where(text['video_id'] == vid)[0]
    if not len(sel):
        return '(video dev, không có caption query)'
    b = max(sel, key=lambda i: tiou(seg, (text['gt_ts'][i], text['gt_te'][i])))
    return f"{text['sentence'][b]}  [{text['gt_ts'][b]:.0f}-{text['gt_te'][b]:.0f}]"

def cut_clip(vid, ts, te):
    os.makedirs('/content/clips', exist_ok=True)
    out = f'/content/clips/{vid}_{ts:.1f}_{te:.1f}.mp4'
    if not os.path.exists(out):
        subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-ss', str(ts), '-to', str(te),
                        '-i', f'{cfg.video_root}/{vid}.mp4', '-vf', 'scale=-2:240',
                        '-c:v', 'libx264', '-preset', 'veryfast', '-c:a', 'aac', out], check=True)
    return out

def show(q_vec, query, gt_video=None):
    hits = index.search(q_vec, top_k=TOP_K)
    print(f'[{PARTITION}] "{query}"' + (f'   (video GT: {gt_video})' if gt_video else ''))
    display(pd.DataFrame([{'hạng': i + 1, 'video_id': h['video_id'], 'điểm': round(h['score'], 4),
                           '[t_s, t_e]': f"[{h['best_segment'][0]:.1f}, {h['best_segment'][1]:.1f}]",
                           'caption GT gần nhất': nearest_gt_caption(h['video_id'], h['best_segment']),
                           'đúng video': 'x' if h['video_id'] == gt_video else ''}
                          for i, h in enumerate(hits)]))
    for h in hits[:SHOW_CLIPS]:
        display(Video(cut_clip(h['video_id'], *h['best_segment']), embed=True, width=420))

if not LOAD_MODEL:
    i = CAPTION_INDEX
    show(text['emb'][i], text['sentence'][i], gt_video=text['video_id'][i])
else:
    from omni_retrieval.encode import encode_texts, load_model
    if 'MODEL' not in globals():   # không bao giờ nạp model lần hai trong một phiên
        MODEL, PROC, _ = load_model(OMNI_REPO, WAVE, BEATS, ADAPTER)
        chk = encode_texts(MODEL, PROC.tokenizer, [str(text['sentence'][0])])[0]
        print('kiểm tra nhánh text: cos(query mới, cache) =', round(float(chk @ text['emb'][0]), 4))
    while True:
        q = input('query (Enter để thoát): ').strip()
        if not q:
            break
        show(encode_texts(MODEL, PROC.tokenizer, [q])[0], q)